# Figura 4 · Los tramos del IRPF, explicados con una nómina (2024)

Este cuaderno crea la Figura 4 del trabajo con **Plotly**, una librería de Python que genera gráficos interactivos. El resultado es un archivo `figura4.html` que se incrusta en la web con un iframe, igual que los gráficos de Datawrapper.

**Qué muestra.** La escalera de la escala general del IRPF de 2024: cada escalón es un tramo de la *base liquidable* y su altura es el tipo que paga cada euro de ese tramo. Con unos botones se elige una nómina (salario mínimo, salario mediano, salario medio o percentil 90) y se colorea la parte de la escalera que ocupa su base: **el área coloreada es el IRPF que paga**. Al pasar el ratón por cada trozo se ve cuántos euros caen en ese tramo y cuánto impuesto pagan.

**Datos.** Los sueldos salen de `data/processed/salarios_ccaa.csv` (fila «Nacional», 2024) y el cálculo del impuesto del modelo fiscal del repositorio (`src/modelo_fiscal.py`), el mismo que usa el cuaderno principal. No se crea ninguna tabla nueva.

**Supuestos** (los del modelo fiscal): asalariado soltero, sin hijos ni otras rentas. La escala autonómica se supone igual a la estatal, así que el tipo de cada tramo es el doble del estatal (en la práctica cada comunidad tiene su propia escala).

## 1. Preparación
Descargamos el repositorio (si no estamos ya dentro de él) para poder usar el modelo fiscal y las tablas de trabajo.

In [1]:
import os, subprocess, sys
import pandas as pd
import plotly.graph_objects as go

REPO = "https://github.com/sortaria24-max/08MBID_TrabajoPractico.git"
if not os.path.exists("src/modelo_fiscal.py"):
    if os.path.exists("../src/modelo_fiscal.py"):   # ejecutado desde la carpeta notebooks/
        os.chdir("..")
    else:                                            # Colab: descargamos el repositorio
        subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
        os.chdir("08MBID_TrabajoPractico")
sys.path.insert(0, "src")
import modelo_fiscal as mf

AÑO = 2024
AZUL, NARANJA, TINTA, TINTA2, GRIS = "#2A78D6", "#EB6834", "#14130F", "#55534C", "#D9D6CC"
# Un color por tramo, de claro a oscuro: cuanto más alto el tramo, más oscuro
COLORES_TRAMO = ["#CFE0F5", "#9DC1EC", "#5F9BE0", "#2A78D6", "#1A4F91", "#0F2F57"]

def euros(x):
    return f"{x:,.0f}".replace(",", ".") + " €"

def pct(x):
    return f"{x:.1f}".replace(".", ",") + " %"

## 2. La escala de 2024

El modelo fiscal guarda los umbrales y los tipos de la escala estatal. El tipo total de cada tramo es el estatal más el autonómico, que aquí es el doble del estatal. El último tramo empieza en 300.000 €, fuera del gráfico.

In [2]:
umbrales = mf.umbrales_escala(AÑO)                       # [0, 12.450, 20.200, 35.200, 60.000, 300.000]
tipos = [2 * t for t in mf.TIPOS_ESTATAL[:len(umbrales)]]   # 19, 24, 30, 37, 45, 47 %
techos = umbrales[1:] + [float("inf")]
escala = pd.DataFrame({"desde": umbrales, "hasta": techos, "tipo_pct": tipos})
escala

,desde,hasta,tipo_pct
0,0,12450.0,19.0
1,12450,20200.0,24.0
2,20200,35200.0,30.0
3,35200,60000.0,37.0
4,60000,300000.0,45.0
5,300000,inf,49.0


## 3. Cuatro nóminas de ejemplo

Para cada sueldo bruto calculamos, con el modelo fiscal, los pasos de la nómina hasta la base sobre la que se aplica la escala:

1. **Sueldo bruto** menos la **Seguridad Social** que paga el trabajador.
2. Menos **2.000 €** de gastos deducibles.
3. Menos la **reducción por rendimientos del trabajo**, que solo tienen los sueldos bajos.
4. El resultado es la **base liquidable**. Sus primeros **5.550 €** son el *mínimo personal*, que no paga impuesto; el resto paga el tipo del tramo en el que cae cada euro.

Comprobamos que la suma de lo que paga cada tramo coincide con el IRPF que calcula el modelo.

In [3]:
s = pd.read_csv("data/processed/salarios_ccaa.csv")
nac = s[(s.ccaa == "Nacional") & (s.anio == AÑO)].iloc[0]
SMI_ANUAL = 1134 * 14                                     # SMI 2024: 1.134 €/mes en 14 pagas

nominas = {"Salario mínimo": SMI_ANUAL, "Salario mediano": nac.mediana,
           "Salario medio": nac.media, "Percentil 90": nac.p90}

def desglose(bruto):
    ss = mf.cotizacion_ss(bruto, AÑO)
    reduccion = mf.reduccion_trabajo(bruto - ss, AÑO)
    base, _ = mf.rendimiento_neto(bruto, AÑO)
    irpf, _ = mf.irpf_individual(bruto, AÑO)
    trozos = []                                           # (desde, hasta, tipo, € de base, € de IRPF)
    for d, h, t in escala.itertuples(index=False):
        ini, fin = max(d, mf.MINIMO_CONTRIBUYENTE), min(h, base)
        if fin > ini:
            trozos.append((ini, fin, t, fin - ini, (fin - ini) * t / 100))
    assert abs(sum(tr[4] for tr in trozos) - irpf) < 0.01, "Los tramos no suman el IRPF del modelo"
    return dict(bruto=bruto, ss=ss, reduccion=reduccion, base=base, irpf=irpf, trozos=trozos)

datos = {n: desglose(b) for n, b in nominas.items()}
pd.DataFrame({n: {"Bruto": d["bruto"], "Seguridad Social": d["ss"], "Gastos": mf.GASTOS_OTROS,
                  "Reducción": d["reduccion"], "Base liquidable": d["base"], "IRPF": d["irpf"],
                  "Tipo medio (%)": 100 * d["irpf"] / d["bruto"]} for n, d in datos.items()}).round(1)

,Salario mínimo,Salario mediano,Salario medio,Percentil 90
Bruto,15876.0,24497.2,29540.3,52515.3
Seguridad Social,1027.2,1585.0,1911.3,3397.7
Gastos,2000.0,2000.0,2000.0,2000.0
Reducción,7302.0,0.0,0.0,0.0
Base liquidable,5546.8,20912.2,25629.0,47117.5
IRPF,0.0,3384.7,4799.7,12080.5
Tipo medio (%),0.0,13.8,16.2,23.0


## 4. Gráfico interactivo

- **Escalera gris:** la escala completa, igual para todos.
- **Trozo rayado:** el mínimo personal, la parte de la base que no paga.
- **Trozos de color:** la parte de la base que cae en cada tramo. Su área es el impuesto que paga ese tramo.
- **Botones:** cambian de nómina. El recuadro de arriba resume los pasos de la nómina hasta el IRPF.

Cada nómina tiene sus propias capas en el gráfico; los botones muestran las de la nómina elegida y ocultan las demás.

In [4]:
XMAX = 70000
fig = go.Figure()

# Escalera de la escala (siempre visible)
xs, ys = [], []
for d, h, t in escala.itertuples(index=False):
    xs += [d, min(h, XMAX)]; ys += [t, t]
fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines", line=dict(color=TINTA2, width=1.5, shape="hv"),
                         fill="tozeroy", fillcolor="rgba(217,214,204,0.35)", hoverinfo="skip", showlegend=False))
for d, h, t in escala.itertuples(index=False):
    if d < XMAX:
        fig.add_annotation(x=(d + min(h, XMAX)) / 2, y=t, text=f"<b>{t:.0f} %</b>", showarrow=False,
                           yshift=12, font=dict(size=13, color=TINTA))

# Capas de cada nómina
capas, resumen = {}, {}
for nombre, d in datos.items():
    ids = []
    m = min(d["base"], mf.MINIMO_CONTRIBUYENTE)                    # mínimo personal
    fig.add_trace(go.Scatter(x=[0, 0, m, m, 0], y=[0, tipos[0], tipos[0], 0, 0], fill="toself",
        fillcolor="rgba(255,255,255,0.6)", fillpattern=dict(shape="/", fgcolor=TINTA2, size=6),
        line=dict(color=TINTA2, width=1), mode="lines", visible=False, hoveron="fills",
        hoverinfo="text", text=f"<b>Mínimo personal</b><br>{euros(m)} de base<br>No pagan IRPF",
        showlegend=False))
    ids.append(len(fig.data) - 1)
    for i, (ini, fin, t, eur, imp) in enumerate(d["trozos"]):
        k = list(escala.tipo_pct).index(t)
        fig.add_trace(go.Scatter(x=[ini, ini, fin, fin, ini], y=[0, t, t, 0, 0], fill="toself",
            fillcolor=COLORES_TRAMO[k], line=dict(color="white", width=1), mode="lines", visible=False,
            hoveron="fills", hoverinfo="text",
            text=f"<b>Tramo del {t:.0f} %</b><br>{euros(eur)} de base<br>IRPF: {euros(imp)}",
            showlegend=False))
        ids.append(len(fig.data) - 1)
    fig.add_trace(go.Scatter(x=[d["base"], d["base"]], y=[0, 50], mode="lines", visible=False,
        line=dict(color=NARANJA, width=2, dash="dash"), hoverinfo="skip", showlegend=False))
    ids.append(len(fig.data) - 1)
    capas[nombre] = ids
    resumen[nombre] = (f"<b>{nombre}: {euros(d['bruto'])} brutos</b><br>"
        f"− Seguridad Social {euros(d['ss'])} − gastos {euros(mf.GASTOS_OTROS)}<br>"
        f"− reducción {euros(d['reduccion'])} = <b>base liquidable {euros(d['base'])}</b><br>"
        f"IRPF: <b>{euros(d['irpf'])}</b> al año, el <b>{pct(100 * d['irpf'] / d['bruto'])}</b> del bruto")

def caja(nombre):
    return dict(x=0, y=1.0, xref="paper", yref="paper", xanchor="left", yanchor="bottom", align="left",
                showarrow=False, text=resumen[nombre], font=dict(size=13, color=TINTA),
                bgcolor="white", bordercolor=GRIS, borderwidth=1, borderpad=8)

anotaciones_fijas = list(fig.layout.annotations)
botones = []
for nombre in datos:
    vis = [True] + [False] * (len(fig.data) - 1)
    for i in capas[nombre]:
        vis[i] = True
    botones.append(dict(label=nombre, method="update",
                        args=[{"visible": vis}, {"annotations": anotaciones_fijas + [caja(nombre)]}]))

INICIAL = "Salario mediano"
for i in capas[INICIAL]:
    fig.data[i].visible = True
fig.update_layout(
    annotations=anotaciones_fijas + [caja(INICIAL)],
    updatemenus=[dict(type="buttons", direction="right", buttons=botones, active=list(datos).index(INICIAL),
                      x=0, y=1.42, xanchor="left", yanchor="bottom", pad=dict(r=6, t=0),
                      bgcolor="white", bordercolor=GRIS, font=dict(size=12, color=TINTA))],
    xaxis=dict(title="Base liquidable (€ al año)", range=[0, XMAX], tickvals=list(range(0, XMAX + 1, 10000)),
               ticktext=[euros(v) if v else "0" for v in range(0, XMAX + 1, 10000)],
               showgrid=False, zeroline=False, linecolor=TINTA2, fixedrange=True),
    yaxis=dict(title="Tipo de cada tramo", range=[0, 52], ticksuffix=" %", gridcolor="#ECEAE3",
               zeroline=False, fixedrange=True),
    font=dict(family="Inter, system-ui, -apple-system, Segoe UI, sans-serif", color=TINTA2, size=12),
    paper_bgcolor="white", plot_bgcolor="white", height=580,
    margin=dict(l=60, r=20, t=190, b=60), separators=",.", hoverlabel=dict(font_size=13),
)
fig.show(config={"displaylogo": False})

## 5. Guardar

- `figura4.html`: el gráfico interactivo, para subirlo a StackBlitz junto a `index.html`. Carga Plotly desde internet, así que el archivo pesa poco.
- Para el PDF: en la esquina superior derecha del gráfico, el icono de la cámara descarga `figura4.png` con la nómina que se esté viendo (el salario mediano, por defecto).

En Colab, los archivos aparecen en el panel de la izquierda (icono de carpeta); esta celda además descarga el HTML.

In [5]:
config = {"displaylogo": False, "responsive": True,
          "toImageButtonOptions": {"format": "png", "filename": "figura4", "scale": 2, "width": 1000, "height": 580}}
fig.write_html("figura4.html", include_plotlyjs="cdn", full_html=True, config=config)

# Misma tipografía que la web
html = open("figura4.html", encoding="utf-8").read()
fuente = ('<link rel="stylesheet" href="https://fonts.googleapis.com/css2?family=Inter:wght@400;600&display=swap">'
          '<style>body{margin:0}</style>')
open("figura4.html", "w", encoding="utf-8").write(html.replace("<head>", "<head>" + fuente, 1))
print("Guardado: figura4.html")

try:
    from google.colab import files
    files.download("figura4.html")
except ImportError:
    pass

Guardado: figura4.html
